# S17 · Parcial NovaMarket — Lote 08

Plantilla reproducible. Las solicitudes de Camila de los minutos 50 y 80 quedan pendientes hasta que se publiquen; no se anticipan ni se inventan sus archivos.

El CSV final conserva unidades reales, no aplica escalado ni codificación. Las columnas categóricas se conservan normalizadas; las variables de S13 se generan con la información disponible y `fuga_cliente` se reserva como objetivo, no como predictor. `id_pedido` e `id_cliente` se conservan solo como llaves de trazabilidad/agrupación; los correos se excluyen y `fecha_llamada_retencion`, posterior al pedido, no se entrega.

## 1. Carga del lote

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd

RAIZ = Path.cwd().resolve()
if not (RAIZ / 'NovaMarket_datos_crudos.csv').exists():
    RAIZ = RAIZ.parent
S17 = RAIZ / 'S17'
sys.path.insert(0, str(S17))
from funciones_novamarket import limpiar_lote, auditar_lote_vs_semestre

ruta_lote = S17 / 'NovaMarket_lote_noviembre_08.csv'
ruta_referencia = S17 / 'metricas_semestre.json'
lote_crudo = pd.read_csv(ruta_lote, low_memory=False)
with ruta_referencia.open(encoding='utf-8') as archivo:
    referencia = json.load(archivo)

print(f'Archivo: {ruta_lote.name}')
print(f'Filas: {len(lote_crudo):,} | columnas: {lote_crudo.shape[1]}')
display(lote_crudo.head())
display(lote_crudo.isna().sum().sort_values(ascending=False).to_frame('vacios crudos'))

Archivo: NovaMarket_lote_noviembre_08.csv
Filas: 20,300 | columnas: 21


,id_pedido,id_cliente,fecha_pedido,canal_compra,metodo_pago,ciudad_tienda,codigo_postal,categoria_producto,producto,precio_unitario,...,monto_compra,edad_cliente,correo_cliente,nivel_satisfaccion,nivel_lealtad,comentario_cliente,peso_pedido_kg,fecha_actualizacion_stock,fecha_llamada_retencion,fuga_cliente
0,N08006763,K0801732,28/10/2026,Tienda,TRANSFERENCIA,CARTAGENA,1300104,Moda,Zapatos deportivos,109000.0,...,109000,57.0,cliente086763@hotmail.com,Alto,Plata,Rápido y sin problemas,1105.7,2024-07-20,NaN,NaN
1,N08013389,K0801203,17/10/2026,marketplace,PayPal,Medellín,500101,Hogar,Set de cuchillos,263600.0,...,263600,58.0,cliente0813389@gmail.com,Bajo,Plata,Llegó incompleto el pedido,11.6,2026-11-06,NaN,NaN
2,N08009122,K0800401,30/10/2026,Marketplace,Transferencia,bogota,1100106,Moda,Bufanda de lana,440200.0,...,$440.200,26.0,cliente089122@outlook.com,Alto,Platino,Llegó incompleto el pedido,13.3,2026-10-13,NaN,NaN
3,N08010331,K0804751,18/10/2026,Tienda,Transferencia,cucuta,5400107,electrónica,Teclado mecánico,28900.0,...,57800,61.0,cliente0810331@yahoo.es,bajo,Plata,Pésima experiencia con el empaque,1048.3,2026-09-20,2026-12-04,NaN
4,N08011547,K0800045,13/11/2027,Web,transferencia,Santa Marta,4700101,Belleza,Secador de pelo,956300.0,...,1912600,32.0,NaN,Bajo,Plata,Llegó incompleto el pedido,8.3,2026-11-02,NaN,NaN


,vacios crudos
fuga_cliente,16246
fecha_llamada_retencion,14552
correo_cliente,3332
comentario_cliente,2889
edad_cliente,2421
nivel_satisfaccion,1471
ciudad_tienda,950
monto_compra,690
precio_unitario,690
id_pedido,0


## 2. Auditoría del lote frente al semestre

Esta primera comparación es sobre el archivo crudo: las categorías se normalizan solo para comparar, así se distinguen variantes de escritura de categorías realmente nuevas. Las diferencias de columnas y tipos se muestran tal como llegan y se vuelven a revisar después del preproceso.

In [2]:
auditoria_cruda = auditar_lote_vs_semestre(
    lote_crudo, referencia=referencia, titulo='archivo crudo'
)
print('\nMedianas de referencia del semestre:')
display(pd.Series(referencia['semestre']['medianas'], name='mediana_semestre').to_frame())
print('\nPorcentajes de vacíos de referencia del semestre:')
display(pd.Series(referencia['semestre']['porcentaje_vacios'], name='porcentaje_semestre').to_frame())


Auditoria archivo crudo frente al semestre (20,300 filas):
Columnas solo en el lote: ['correo_cliente', 'fecha_llamada_retencion']
Columnas solo en el semestre: ['acepta_promociones', 'antiguedad_cliente_dias', 'bodega_asignada', 'bodega_encontrada', 'categoria', 'ciudad_residencia', 'ciudad_tienda_imputado', 'comentario_cliente_imputado', 'correo_cliente_imputado', 'crm_encontrada', 'dias_desde_compra_previa', 'dias_reposicion', 'edad_cliente_imposible', 'es_venta_corporativa', 'fecha_inscripcion_club', 'gasto_previo_cliente', 'monto_compra_capado', 'monto_compra_recalculado', 'monto_por_unidad', 'nivel_satisfaccion_imputado', 'pedidos_previos_cliente', 'peso_pedido_kg_imposible', 'precio_monto_imputado', 'proveedor_principal', 'puntos_fidelidad_actuales', 'rfm_score', 'segmento_crm', 'stock_minimo', 'ticket_promedio_previo_cliente']
Diferencias de tipos (lote, semestre): {'canal_compra': ('object', 'string'), 'categoria_producto': ('object', 'string'), 'ciudad_tienda': ('object', 's

Diferencias de categorias: {'canal_compra': {'nuevas_en_lote': ['TikTok Shop'], 'no_observadas_en_lote': []}, 'metodo_pago': {'nuevas_en_lote': ['Contraentrega'], 'no_observadas_en_lote': []}}
Diferencias de medianas (lote vs. semestre): {'precio_unitario': {'mediana_lote': 355500.0, 'mediana_semestre': 305600.0, 'diferencia': 49900.0}, 'unidades_vendidas': {'mediana_lote': 1.0, 'mediana_semestre': 2.0, 'diferencia': -1.0}, 'monto_compra': {'mediana_lote': 527300.0, 'mediana_semestre': 476400.0, 'diferencia': 50900.0}, 'edad_cliente': {'mediana_lote': 47.0, 'mediana_semestre': 46.0, 'diferencia': 1.0}, 'peso_pedido_kg': {'mediana_lote': 11.7, 'mediana_semestre': 11.5, 'diferencia': 0.1999999999999993}}
Diferencias de porcentaje de vacios (puntos porcentuales): {'id_pedido': {'porcentaje_lote': 0.0, 'porcentaje_semestre': 0.0, 'diferencia_pp': 0.0}, 'id_cliente': {'porcentaje_lote': 0.0, 'porcentaje_semestre': 0.0, 'diferencia_pp': 0.0}, 'fecha_pedido': {'porcentaje_lote': 0.0, 'porcent

,mediana_semestre
precio_unitario,305600.0
unidades_vendidas,2.0
monto_compra,476400.0
edad_cliente,46.0
peso_pedido_kg,11.5



Porcentajes de vacíos de referencia del semestre:


,porcentaje_semestre
id_pedido,0.0000
id_cliente,0.0000
fecha_pedido,3.5490
canal_compra,0.0000
metodo_pago,0.0000
ciudad_tienda,4.6686
codigo_postal,0.0000
categoria_producto,0.0000
producto,0.0000
precio_unitario,0.0000


## 3. Preproceso

`limpiar_lote` elimina duplicados exactos, convierte y reporta conversiones que crean vacíos, normaliza texto, trata valores imposibles y faltantes, integra CRM/bodega con validación `m:1` y crea las variables de S13. No se crean columnas codificadas de S09 porque el entregable exige categorías legibles y unidades originales.

In [3]:
lote_limpio = limpiar_lote(lote_crudo)
ruta_salida = S17 / 'S17_PD_PallaresArbelaezSanchez_Lote08_preprocesado.csv'
lote_limpio.to_csv(ruta_salida, index=False, encoding='utf-8-sig')
print(f'Entregable escrito: {ruta_salida.name} ({len(lote_limpio):,} filas)')

Lote crudo: 20,300 filas, 21 columnas.
Duplicados exactos: 300; filas tras eliminarlos: 20,000.
Conversion id_pedido: 0 valores no vacios pasaron a vacios.


Conversion id_cliente: 0 valores no vacios pasaron a vacios.
Conversion codigo_postal: 0 valores no vacios pasaron a vacios.


Conversion precio_unitario: 0 valores no vacios pasaron a vacios.
Conversion unidades_vendidas: 0 valores no vacios pasaron a vacios.
Conversion edad_cliente: 0 valores no vacios pasaron a vacios.
Conversion peso_pedido_kg: 0 valores no vacios pasaron a vacios.
Conversion monto_compra: 0 valores no vacios pasaron a vacios.


Conversion fecha_pedido: 710 valores no vacios pasaron a vacios.
Conversion fecha_actualizacion_stock: 0 valores no vacios pasaron a vacios.
Conversion fecha_llamada_retencion: 0 valores no vacios pasaron a vacios.
Conversion fuga_cliente: 0 valores no vacios pasaron a vacios.


Normalizacion canal_compra: 0 vacios nuevos; 5 categorias.


Normalizacion metodo_pago: 0 vacios nuevos; 5 categorias.


... (5 outputs left out)

Integracion bodega: filas antes=17,804, despues=17,804; llaves sin pareja=2,946; llaves repetidas en tabla=0.


Features de historial calculadas solo dentro del lote: 17,169 filas con fecha/cliente.
Columna excluida por ser posterior al pedido: fecha_llamada_retencion.
Columnas personales usadas solo para CRM y excluidas de la salida: correo_cliente, email_cliente.


Limpieza finalizada: 17,804 filas (inicio 20,300), 48 columnas; pedidos unicos=17,804.


Entregable escrito: S17_PD_PallaresArbelaezSanchez_Lote08_preprocesado.csv (17,804 filas)


## 4. Verificación del resultado

Se verifica el lote real y, además, una muestra reproducible de 5.000 filas del CSV crudo del semestre procesada por la misma función.

In [4]:
def verificar_resultado(df, nombre):
    print(f'\n{nombre}: {len(df):,} filas, {df.shape[1]} columnas')
    print('Duplicados exactos:', int(df.duplicated().sum()))
    print('id_pedido repetidos:', int(df['id_pedido'].duplicated().sum()))
    print('Tipos:')
    display(df.dtypes.astype(str).to_frame('dtype'))
    print('Vacíos:')
    display(df.isna().sum().sort_values(ascending=False).to_frame('cantidad'))
    for columna in ['canal_compra', 'metodo_pago', 'categoria_producto', 'ciudad_tienda', 'nivel_satisfaccion', 'nivel_lealtad']:
        if columna in df.columns:
            print(f'{columna}: {sorted(df[columna].dropna().astype(str).unique())}')
    assert df['id_pedido'].is_unique
    assert df.duplicated().sum() == 0
    assert pd.api.types.is_datetime64_any_dtype(df['fecha_pedido'])
    assert pd.api.types.is_integer_dtype(df['unidades_vendidas'])
    assert pd.api.types.is_numeric_dtype(df['precio_unitario'])
    assert pd.api.types.is_numeric_dtype(df['monto_compra'])
    assert df['unidades_vendidas'].gt(0).all()

verificar_resultado(lote_limpio, 'Lote noviembre limpio')
auditoria_limpia = auditar_lote_vs_semestre(
    lote_limpio, referencia=referencia, titulo='lote limpio'
)

crudo_semestre = pd.read_csv(RAIZ / 'NovaMarket_datos_crudos.csv', low_memory=False)
clientes_lote = set(lote_limpio['id_cliente'].dropna().astype(str))
clientes_semestre = set(crudo_semestre['id_cliente'].dropna().astype(str))
clientes_compartidos = len(clientes_lote & clientes_semestre)
print(f'Clientes con historial en el semestre: {clientes_compartidos:,} de {len(clientes_lote):,} del lote.')
muestra_cruda = crudo_semestre.sample(n=5000, random_state=42).copy()
print(f'\nMuestra cruda del semestre: {len(muestra_cruda):,} filas; semilla=42')
muestra_limpia = limpiar_lote(muestra_cruda)
verificar_resultado(muestra_limpia, 'Muestra del semestre limpia')
auditoria_muestra = auditar_lote_vs_semestre(
    muestra_limpia, referencia=referencia, titulo='muestra de 5.000 filas limpia'
)
print('Columnas nuevas o ausentes frente al esquema de referencia:')
print(auditoria_limpia['columnas_solo_lote'], auditoria_limpia['columnas_solo_semestre'])


Lote noviembre limpio: 17,804 filas, 48 columnas


Duplicados exactos: 0
id_pedido repetidos: 0
Tipos:


,dtype
id_pedido,string
id_cliente,string
fecha_pedido,datetime64[ns]
canal_compra,string
metodo_pago,string
ciudad_tienda,string
codigo_postal,string
categoria_producto,string
producto,string
precio_unitario,float64


Vacíos:


,cantidad
segmento_crm,17804
fecha_inscripcion_club,17804
antiguedad_cliente_dias,17804
acepta_promociones,17804
ciudad_residencia,17804
puntos_fidelidad_actuales,17804
fuga_cliente,14249
ticket_promedio_previo_cliente,5721
dias_desde_compra_previa,5721
categoria,2946


canal_compra: ['App', 'Marketplace', 'Tienda', 'TikTok Shop', 'Web']
metodo_pago: ['Contraentrega', 'Efectivo', 'PayPal', 'Tarjeta', 'Transferencia']
categoria_producto: ['Belleza', 'Deportes', 'Electronica', 'Hogar', 'Jugueteria', 'Moda']
ciudad_tienda: ['Barranquilla', 'Bogota', 'Bucaramanga', 'Cali', 'Cartagena', 'Cucuta', 'Ibague', 'Manizales', 'Medellin', 'Pereira', 'Santa Marta', 'Villavicencio']
nivel_satisfaccion: ['Alto', 'Bajo', 'Medio']
nivel_lealtad: ['Bronce', 'Oro', 'Plata', 'Platino']


... (12 outputs left out)

Vacíos:


,cantidad
ticket_promedio_previo_cliente,4426
dias_desde_compra_previa,4426
antiguedad_cliente_dias,2569
acepta_promociones,1528
puntos_fidelidad_actuales,1528
fecha_inscripcion_club,1528
ciudad_residencia,1528
segmento_crm,1528
bodega_asignada,782
dias_reposicion,782


canal_compra: ['App', 'Marketplace', 'Tienda', 'Web']
metodo_pago: ['Efectivo', 'PayPal', 'Tarjeta', 'Transferencia']
categoria_producto: ['Belleza', 'Deportes', 'Electronica', 'Hogar', 'Jugueteria', 'Moda']
ciudad_tienda: ['Barranquilla', 'Bogota', 'Bucaramanga', 'Cali', 'Cartagena', 'Cucuta', 'Ibague', 'Manizales', 'Medellin', 'Pereira', 'Santa Marta', 'Villavicencio']
nivel_satisfaccion: ['Alto', 'Bajo', 'Medio']
nivel_lealtad: ['Bronce', 'Oro', 'Plata', 'Platino']

Auditoria muestra de 5.000 filas limpia frente al semestre (4,489 filas):
Columnas solo en el lote: ninguna
Columnas solo en el semestre: ninguna
Diferencias de tipos (lote, semestre): ninguna


Diferencias de categorias: ninguna
Diferencias de medianas (lote vs. semestre): {'precio_unitario': {'mediana_lote': 305600.0, 'mediana_semestre': 305600.0, 'diferencia': 0.0}, 'unidades_vendidas': {'mediana_lote': 1.0, 'mediana_semestre': 2.0, 'diferencia': -1.0}, 'monto_compra': {'mediana_lote': 477900.0, 'mediana_semestre': 476400.0, 'diferencia': 1500.0}, 'edad_cliente': {'mediana_lote': 47.0, 'mediana_semestre': 46.0, 'diferencia': 1.0}, 'peso_pedido_kg': {'mediana_lote': 11.5, 'mediana_semestre': 11.5, 'diferencia': 0.0}}
Diferencias de porcentaje de vacios (puntos porcentuales): {'id_pedido': {'porcentaje_lote': 0.0, 'porcentaje_semestre': 0.0, 'diferencia_pp': 0.0}, 'id_cliente': {'porcentaje_lote': 0.0, 'porcentaje_semestre': 0.0, 'diferencia_pp': 0.0}, 'fecha_pedido': {'porcentaje_lote': 3.6088215638226777, 'porcentaje_semestre': 3.549, 'diferencia_pp': 0.05982156382267778}, 'canal_compra': {'porcentaje_lote': 0.0, 'porcentaje_semestre': 0.0, 'diferencia_pp': 0.0}, 'metodo_pa

## 5. Solicitudes de Camila (vacía)

### Minuto 50
- Mensaje recibido: **pendiente**.
- Archivo recibido y nombre exacto de salida: **pendiente**.
- Preproceso, evidencia, decisión y limitaciones: **completar cuando se publique la solicitud**.

### Minuto 80
- Mensaje recibido: **pendiente**.
- Archivo recibido y nombre exacto de salida: **pendiente**.
- Preproceso, evidencia, decisión y limitaciones: **completar cuando se publique la solicitud**.

## 6. Acta de incidentes

La celda siguiente prioriza hallazgos de impacto y registra las cinco partes por incidente. Cada evidencia se calcula desde las variables del lote; se revisa el acta cuando lleguen los archivos de Camila.

In [5]:
n = len(lote_crudo)
duplicados = int(lote_crudo.duplicated().sum())
filas_eliminadas_por_imposibles = max(0, n - duplicados - len(lote_limpio))
edad_imposible = int(lote_limpio['edad_cliente_imposible'].sum())
peso_imposible = int(lote_limpio['peso_pedido_kg_imposible'].sum())
crm_sin_pareja = int((~lote_limpio['crm_encontrada']).sum())
bodega_sin_pareja = int((~lote_limpio['bodega_encontrada']).sum())
montos_capados = int(lote_limpio['monto_compra_capado'].sum())
ventas_corporativas = int(lote_limpio['es_venta_corporativa'].sum())
clientes_lote_total = lote_limpio['id_cliente'].nunique()
clientes_semestre_total = len(clientes_semestre)
fechas_pedido_invalidas = int(lote_limpio['fecha_pedido'].isna().sum())
etiquetas_fuga_vacias = int(lote_limpio['fuga_cliente'].isna().sum()) if 'fuga_cliente' in lote_limpio else 0
nuevas_categorias = {}
for columna, categorias_semestre in referencia['semestre']['categorias'].items():
    nuevas = set(lote_limpio[columna].dropna().astype(str)) - set(categorias_semestre)
    for categoria in sorted(nuevas):
        cantidad = int(lote_limpio[columna].eq(categoria).sum())
        nuevas_categorias[f'{columna}={categoria}'] = cantidad

incidentes = [
    {
        'impacto': 1,
        'qué encontraste': 'Montos altos modificados por el límite IQR',
        'evidencia': f'{montos_capados:,} montos capados y {ventas_corporativas:,} ventas corporativas marcadas; límite semestral {referencia["limite_superior_iqr_monto"]:,.0f} pesos.',
        'decisión': 'Aplicar la regla S07: capar 2-3 unidades y conservar/etiquetar 4-5 unidades.',
        'por qué': 'Se mantiene la política documentada del semestre y no se descartan ventas de alto valor.',
        'riesgo que queda': 'El cap modifica ingresos; conciliar esos pedidos con el CSV crudo antes de un informe financiero.'
    },
    {
        'impacto': 2,
        'qué encontraste': 'Etiqueta fuga_cliente incompleta',
        'evidencia': f'{etiquetas_fuga_vacias:,} de {len(lote_limpio):,} ({etiquetas_fuga_vacias / max(len(lote_limpio), 1):.2%}) no tienen etiqueta.',
        'decisión': 'Conservar los valores conocidos como objetivo y no imputar los faltantes.',
        'por qué': 'La etiqueta representa un resultado y no debe inferirse a partir de predictores.',
        'riesgo que queda': 'No entrenar ni evaluar un modelo supervisado de fuga hasta aclarar la cobertura.'
    },
    {
        'impacto': 3,
        'qué encontraste': 'Pedidos con valores de venta imposibles',
        'evidencia': f'{filas_eliminadas_por_imposibles:,} filas ({filas_eliminadas_por_imposibles / max(n, 1):.2%} del lote), aparte de {duplicados:,} duplicados exactos; reglas S03/S07.',
        'decisión': 'Excluir filas con precio negativo/centinela, unidades no positivas/centinela o monto negativo.',
        'por qué': 'No representan una venta interpretable según las reglas de calidad ya documentadas.',
        'riesgo que queda': 'Un pedido real mal capturado podría haber sido excluido; revisar con la fuente antes de reportar ingresos.'
    },
    {
        'impacto': 6,
        'qué encontraste': 'Categorías que no existían en el semestre',
        'evidencia': f'{len(nuevas_categorias)} valores nuevos; conteos por valor: {nuevas_categorias}.',
        'decisión': 'Conservar cada valor nuevo en forma canónica; no mapearlo a otra categoría.',
        'por qué': 'Puede ser un canal o medio de pago legítimo introducido en noviembre.',
        'riesgo que queda': 'Los reportes históricos y reglas de imputación pueden no conocer estas categorías.'
    },
    {
        'impacto': 4,
        'qué encontraste': 'Pedidos sin pareja en CRM',
        'evidencia': f'{crm_sin_pareja:,} de {len(lote_limpio):,} ({crm_sin_pareja / max(len(lote_limpio), 1):.2%}); {int(lote_limpio["crm_encontrada"].sum()):,} con pareja.',
        'decisión': 'Conservar pedidos mediante left join y dejar las variables CRM vacías.',
        'por qué': 'La unión validada m:1 conserva una fila por pedido sin inventar datos del cliente.',
        'riesgo que queda': 'Las features de cliente no son comparables para filas sin CRM.'
    },
    {
        'impacto': 5,
        'qué encontraste': 'Sin clientes compartidos con el semestre',
        'evidencia': f'{clientes_compartidos:,} de {clientes_lote_total:,} clientes del lote aparecen entre {clientes_semestre_total:,} clientes históricos.',
        'decisión': 'Interpretar las features históricas como historial dentro del lote solamente.',
        'por qué': 'Los ids de cliente no se emparejan con los del semestre y el CRM tampoco encontró pareja.',
        'riesgo que queda': 'RFM, frecuencia y gasto previo no representan el historial completo del cliente.'
    },
    {
        'impacto': 7,
        'qué encontraste': 'Fechas de pedido inválidas',
        'evidencia': f'{fechas_pedido_invalidas:,} de {len(lote_limpio):,} ({fechas_pedido_invalidas / max(len(lote_limpio), 1):.2%}) quedaron vacías tras convertir.',
        'decisión': 'Conservar pedidos y dejar la fecha como NaT; no adivinar una fecha.',
        'por qué': 'Sin fecha no se pueden ordenar pedidos ni calcular features temporales con evidencia.',
        'riesgo que queda': 'Excluir estas filas de análisis temporal, recencia e historial.'
    },
    {
        'impacto': 8,
        'qué encontraste': 'Categorías sin pareja en catálogo de bodega',
        'evidencia': f'{bodega_sin_pareja:,} de {len(lote_limpio):,} ({bodega_sin_pareja / max(len(lote_limpio), 1):.2%}); {int(lote_limpio["bodega_encontrada"].sum()):,} con pareja.',
        'decisión': 'Conservar el pedido y no asignar bodega/proveedor sin evidencia.',
        'por qué': 'El catálogo no cubre todas las categorías del lote.',
        'riesgo que queda': 'Planificación de inventario incompleta para las categorías sin pareja.'
    },
    {
        'impacto': 9,
        'qué encontraste': 'Edad o peso fuera del rango físico revisado',
        'evidencia': f'{edad_imposible:,} edades y {peso_imposible:,} pesos imposibles entre {len(lote_limpio):,} pedidos conservados.',
        'decisión': 'Convertir solo esos valores a faltante y mantener indicadores de origen.',
        'por qué': 'El pedido sigue siendo útil para ventas; edad y peso no se corrigen por conjetura.',
        'riesgo que queda': 'Los análisis demográficos/logísticos deben excluir o tratar explícitamente esos faltantes.'
    },
    {
        'impacto': 10,
        'qué encontraste': 'Duplicados exactos',
        'evidencia': f'{duplicados:,} de {n:,} filas crudas ({duplicados / max(n, 1):.2%}); quedan {len(lote_limpio):,} pedidos únicos.',
        'decisión': 'Conservar una copia idéntica por pedido; no eliminar compras por correo repetido.',
        'por qué': 'Las copias exactas duplican un evento; correos repetidos pueden ser compras recurrentes.',
        'riesgo que queda': 'Un duplicado no exacto con id_pedido distinto requiere validación externa.'
    },
    {
        'impacto': 11,
        'qué encontraste': 'Valores de ciudad no registrados',
        'evidencia': f'{int(lote_limpio["ciudad_tienda"].isna().sum()):,} vacíos ({lote_limpio["ciudad_tienda"].isna().mean():.2%}) entre {len(lote_limpio):,} pedidos; se compara con el semestre en la tabla de vacíos.',
        'decisión': 'Dejar ciudad vacía y conservar ciudad_tienda_imputado.',
        'por qué': 'S04 dejó esta variable geográfica sin imputar para no inventar residencia/origen.',
        'riesgo que queda': 'Los análisis geográficos representan solo los pedidos con ciudad observada.'
    }
]
acta = pd.DataFrame(incidentes).sort_values('impacto').drop(columns='impacto')
display(acta)

print(
    f'Recomendación: lote con {len(lote_limpio):,} pedidos únicos de {n:,} filas crudas. '
    f'Puede usarse solo para análisis descriptivo de ventas después de documentar las '
    f'{filas_eliminadas_por_imposibles:,} exclusiones por valores imposibles, '
    f'conciliar los {montos_capados:,} montos capados con el archivo crudo para cualquier total financiero, '
    f'validar las categorías nuevas y tratar por separado las filas sin CRM/bodega. '
    f'No entrenar un modelo con {etiquetas_fuga_vacias:,} etiquetas faltantes ni usar variables posteriores al pedido como predictores; '
    f'además, solo {clientes_compartidos:,} clientes del lote tienen id compartido con el semestre; '
    'antes del próximo lote, completar etiquetas, actualizar catálogos y revisar la cobertura de llaves.'
)

print('\nBitácora de IA')
print('Herramientas: Copilot SDK en VS Code para contrastar reglas de S03-S13 y preparar la función/notebook; Python/pandas para ejecutar conversiones, conteos y comprobaciones reproducibles.')
print('Propuesta no adoptada: imputar ciudad_tienda con la moda. Verificación: S04 deja la ciudad sin imputar y crea ciudad_tienda_imputado; se conserva ese criterio para no asignar una geografía sin evidencia.')
print('Decisión aplicada: conservar ciudad faltante con indicador. El equipo debe revisar y aprobar los criterios y completar las solicitudes de Camila cuando aparezcan.')

,qué encontraste,evidencia,decisión,por qué,riesgo que queda
0,Montos altos modificados por el límite IQR,844 montos capados y 995 ventas corporativas m...,Aplicar la regla S07: capar 2-3 unidades y con...,Se mantiene la política documentada del semest...,El cap modifica ingresos; conciliar esos pedid...
1,Etiqueta fuga_cliente incompleta,"14,249 de 17,804 (80.03%) no tienen etiqueta.",Conservar los valores conocidos como objetivo ...,La etiqueta representa un resultado y no debe ...,No entrenar ni evaluar un modelo supervisado d...
2,Pedidos con valores de venta imposibles,"2,196 filas (10.82% del lote), aparte de 300 d...","Excluir filas con precio negativo/centinela, u...",No representan una venta interpretable según l...,Un pedido real mal capturado podría haber sido...
4,Pedidos sin pareja en CRM,"17,804 de 17,804 (100.00%); 0 con pareja.",Conservar pedidos mediante left join y dejar l...,La unión validada m:1 conserva una fila por pe...,Las features de cliente no son comparables par...
5,Sin clientes compartidos con el semestre,"0 de 4,897 clientes del lote aparecen entre 15...",Interpretar las features históricas como histo...,Los ids de cliente no se emparejan con los del...,"RFM, frecuencia y gasto previo no representan ..."
3,Categorías que no existían en el semestre,2 valores nuevos; conteos por valor: {'canal_c...,Conservar cada valor nuevo en forma canónica; ...,Puede ser un canal o medio de pago legítimo in...,Los reportes históricos y reglas de imputación...
6,Fechas de pedido inválidas,"635 de 17,804 (3.57%) quedaron vacías tras con...",Conservar pedidos y dejar la fecha como NaT; n...,Sin fecha no se pueden ordenar pedidos ni calc...,"Excluir estas filas de análisis temporal, rece..."
7,Categorías sin pareja en catálogo de bodega,"2,946 de 17,804 (16.55%); 14,858 con pareja.",Conservar el pedido y no asignar bodega/provee...,El catálogo no cubre todas las categorías del ...,Planificación de inventario incompleta para la...
8,Edad o peso fuera del rango físico revisado,"659 edades y 900 pesos imposibles entre 17,804...",Convertir solo esos valores a faltante y mante...,El pedido sigue siendo útil para ventas; edad ...,Los análisis demográficos/logísticos deben exc...
9,Duplicados exactos,"300 de 20,300 filas crudas (1.48%); quedan 17,...",Conservar una copia idéntica por pedido; no el...,Las copias exactas duplican un evento; correos...,Un duplicado no exacto con id_pedido distinto ...


Recomendación: lote con 17,804 pedidos únicos de 20,300 filas crudas. Puede usarse solo para análisis descriptivo de ventas después de documentar las 2,196 exclusiones por valores imposibles, conciliar los 844 montos capados con el archivo crudo para cualquier total financiero, validar las categorías nuevas y tratar por separado las filas sin CRM/bodega. No entrenar un modelo con 14,249 etiquetas faltantes ni usar variables posteriores al pedido como predictores; además, solo 0 clientes del lote tienen id compartido con el semestre; antes del próximo lote, completar etiquetas, actualizar catálogos y revisar la cobertura de llaves.

Bitácora de IA
Herramientas: Copilot SDK en VS Code para contrastar reglas de S03-S13 y preparar la función/notebook; Python/pandas para ejecutar conversiones, conteos y comprobaciones reproducibles.
Propuesta no adoptada: imputar ciudad_tienda con la moda. Verificación: S04 deja la ciudad sin imputar y crea ciudad_tienda_imputado; se conserva ese criterio p